# Fake News Detector & Generator — Training Notebook

This notebook builds the **detector** model from scratch on the corrected dataset.

**Key design decision (learned from a first, buggy version of this project):**
The raw Kaggle/ISOT Fake/True news dataset lets a model "cheat" — almost all
REAL articles carry a `(Reuters)` wire-service dateline and curly apostrophes,
while FAKE articles carry site-branding junk (`21st Century Wire`, `Featured
Image`, `Getty`) and have contractions already split by plain spaces. A model
trained without removing these hits 99%+ accuracy by detecting **formatting**,
not content — and then fails on real-world typed text.

This notebook removes those shortcuts before training, so the resulting
model actually learns from content.

## 1. Imports

In [ ]:
import re
import pickle
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

STOP_WORDS = set(ENGLISH_STOP_WORDS)

## 2. Load the dataset

In [ ]:
true_df = pd.read_csv("../dataset/True.csv")
fake_df = pd.read_csv("../dataset/Fake.csv")

# Label convention used throughout this project: 1 = Fake, 0 = Real
true_df["label"] = 0
fake_df["label"] = 1

print("Real articles:", len(true_df))
print("Fake articles:", len(fake_df))

true_df.head()

## 3. Exploratory check — why the raw dataset is dangerous

Before cleaning anything, look at how strongly a few *formatting* details
(not content) already separate the two classes almost perfectly.

In [ ]:
print("Pct REAL articles containing '(Reuters)':", true_df['text'].str.contains(r'\(Reuters\)', regex=True).mean())
print("Pct FAKE articles containing '(Reuters)':", fake_df['text'].str.contains(r'\(Reuters\)', regex=True).mean())
print()
print("Pct REAL articles containing curly apostrophe (’):", true_df['text'].str.contains('’').mean())
print("Pct FAKE articles containing curly apostrophe (’):", fake_df['text'].str.contains('’').mean())

These numbers are close to 100% vs 0% — meaning a model doesn't even need
to understand English to separate these classes, it just needs to check
for a Reuters tag or a certain punctuation style. That's the leak we fix
in the next section.

## 4. Text cleaning (leak-proof)

In [ ]:
def clean_text(text: str) -> str:
    text = str(text).lower()

    # --- Remove dataset-specific "shortcut" signals ---
    text = re.sub(r"^[a-z\s,\.\-]+\(reuters\)\s*-\s*", "", text)
    text = re.sub(r"\breuters\b", "", text)
    text = re.sub(r"21st century wire", "", text)
    text = re.sub(r"21stcenturywire\.com", "", text)
    text = re.sub(r"featured image", "", text)
    text = re.sub(r"\bgetty\b", "", text)
    text = re.sub(r"image via", "", text)
    text = re.sub(r"via twitter", "", text)
    text = re.sub(r"read more .*? at:.*", "", text)
    text = re.sub(r"pic\.twitter\.com\S*", "", text)

    # --- Standard cleaning ---
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\d+", "", text)

    # Normalize ALL punctuation (ascii + unicode smart quotes/dashes) to a
    # space, so contractions split consistently regardless of which
    # apostrophe style the source file used.
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    words = [w for w in text.split() if w not in STOP_WORDS and len(w) > 1]
    return " ".join(words)

# quick test
print(clean_text("WASHINGTON (Reuters) - Trump’s new policy, announced Tuesday, faced criticism."))

> **Important:** this exact function also lives in `backend/nlp_utils.py` and is
> imported by `app.py` at prediction time. Training and serving must always
> clean text the same way, or predictions will be wrong even with a perfectly
> good model — this was the root cause of bugs in the first version of this
> project.

## 5. Build the dataset and clean it

In [ ]:
df = pd.concat([true_df, fake_df], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

t0 = time.time()
df["clean_text"] = (df["title"].fillna("") + " " + df["text"].fillna("")).apply(clean_text)
print(f"Cleaned {len(df)} articles in {time.time()-t0:.1f}s")

df[["clean_text", "label"]].head()

## 6. Class balance check

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(x="label", data=df)
plt.xticks([0,1], ["Real","Fake"])
plt.title("Class distribution")
plt.show()

print(df["label"].value_counts(normalize=True))

## 7. Train / test split + TF-IDF vectorization

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 1), min_df=8, max_df=0.6)
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

print("Vocabulary size:", len(tfidf.get_feature_names_out()))
print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

## 8. Train and compare models

We deliberately use **regularized** models (not maximum-depth trees) — a
model that hits 100% train accuracy on a bag-of-words is memorizing, not
learning. A small train/test gap matters more than a perfect score.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=0.05),
    "Random Forest": RandomForestClassifier(
        n_estimators=150, max_depth=12, min_samples_leaf=5, random_state=42, n_jobs=-1
    ),
}

results = {}
for name, clf in models.items():
    t0 = time.time()
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    results[name] = (clf, acc)
    print(f"{name}: test accuracy = {acc:.4f}  (train accuracy = {clf.score(X_train, y_train):.4f})  [{time.time()-t0:.1f}s]")

## 9. Pick the best model

We prefer **Logistic Regression** unless Random Forest is meaningfully
better (>1% higher accuracy) — because Logistic Regression's coefficients
let us show *which direction* a word pushed the prediction (real vs fake),
which powers the "Explain Results" feature. Random Forest's
`feature_importances_` can only say a word *mattered*, not which way.

In [ ]:
best_name = max(results, key=lambda k: results[k][1])
best_model, best_acc = results[best_name]

lr_model, lr_acc = results["Logistic Regression"]
if best_name != "Logistic Regression" and (best_acc - lr_acc) < 0.01:
    best_name, best_model, best_acc = "Logistic Regression", lr_model, lr_acc
    print("Switched to Logistic Regression (near-equal accuracy, better explainability)")

print(f"\nBest model: {best_name}")
print(f"Test accuracy: {best_acc:.4f}")
print(f"Train accuracy: {best_model.score(X_train, y_train):.4f}")

## 10. Evaluation

In [ ]:
pred = best_model.predict(X_test)
print(classification_report(y_test, pred, target_names=["Real", "Fake"]))

cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Real","Fake"], yticklabels=["Real","Fake"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## 11. Sanity check — the test that actually matters

Test-set accuracy can still be optimistic since the test set comes from
the same two source files. The real test is: does the model behave
sensibly on text that has **none** of this dataset's fingerprints? A
healthy model should sit close to 50/50 on a neutral sentence, not
confidently call everything fake (or everything real).

In [ ]:
sample = "The government announced a new education policy to improve higher education access for students across the country this year."
vec = tfidf.transform([clean_text(sample)])
pred = best_model.predict(vec)[0]
proba = best_model.predict_proba(vec)[0]

print("Prediction:", "Fake" if pred == 1 else "Real")
print(f"Confidence: {proba.max()*100:.1f}%")
print("(Close to 50% is healthy here — the sentence has no strong signal either way)")

## 12. Save the model + vectorizer for the API

In [ ]:
with open("../backend/best_model.pkl", "wb") as f:
    pickle.dump(best_model, f)
with open("../backend/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

print("Saved best_model.pkl and tfidf_vectorizer.pkl to ../backend/")

## 13. Next steps

- The **detector** (this notebook) is done — `backend/app.py` loads
  `best_model.pkl` and `tfidf_vectorizer.pkl` and serves `/predict` and
  `/explain`.
- The **generator** (Generative AI half of the project) lives in
  `backend/generate_utils.py` and uses a small local model (`distilgpt2`)
  to generate sample news text via the `/generate` endpoint — no API key
  needed.
- Run `uvicorn app:app --reload` inside `backend/` to start the API, then
  open `frontend/index.html`.